In [47]:
pip install -q pypdf faiss-cpu sentence-transformers google-genai gradio

In [48]:
import os
import numpy as np
import faiss
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai

In [49]:
from getpass import getpass
import os

gemini_api_key = getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = gemini_api_key

Enter your Gemini API key: ··········


In [50]:
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

In [51]:
getpass()

··········


'AQ.Ab8RN6LATAQIhFtWf4Tf9LXISSuGCTDundqJpgCurIiLd-5KHw'

In [52]:
college_text = """
COLLEGE INFORMATION

Attendance Policy:
Students must maintain the minimum attendance percentage
required by the college to be eligible for semester examinations.

Examination:
Semester examinations are conducted at the end of each semester.
Students must complete the required academic and attendance
requirements before appearing for examinations.

Library:
The college library provides textbooks, reference books,
journals and digital learning resources for students.

Placement:
The placement cell conducts aptitude training, technical training,
communication training and interview preparation for students.

Computer Science Department:
The Computer Science department offers courses in programming,
database management, data structures, artificial intelligence,
machine learning and software engineering.

Laboratory:
Students must follow laboratory safety rules and complete
assigned practical exercises during laboratory sessions.

Leave:
Students should obtain appropriate permission for planned leave
according to college regulations.
"""

with open("college_knowledge.txt", "w") as f:
    f.write(college_text)

print("Knowledge document created successfully!")

Knowledge document created successfully!


In [53]:
with open("college_knowledge.txt", "r")as f:
  text= f.read()
  print (text)


COLLEGE INFORMATION

Attendance Policy:
Students must maintain the minimum attendance percentage
required by the college to be eligible for semester examinations.

Examination:
Semester examinations are conducted at the end of each semester.
Students must complete the required academic and attendance
requirements before appearing for examinations.

Library:
The college library provides textbooks, reference books,
journals and digital learning resources for students.

Placement:
The placement cell conducts aptitude training, technical training,
communication training and interview preparation for students.

Computer Science Department:
The Computer Science department offers courses in programming,
database management, data structures, artificial intelligence,
machine learning and software engineering.

Laboratory:
Students must follow laboratory safety rules and complete
assigned practical exercises during laboratory sessions.

Leave:
Students should obtain appropriate permission for p

In [54]:
text= text.replace("\n", " ")
text= " ".join(text.split())
print(text)

COLLEGE INFORMATION Attendance Policy: Students must maintain the minimum attendance percentage required by the college to be eligible for semester examinations. Examination: Semester examinations are conducted at the end of each semester. Students must complete the required academic and attendance requirements before appearing for examinations. Library: The college library provides textbooks, reference books, journals and digital learning resources for students. Placement: The placement cell conducts aptitude training, technical training, communication training and interview preparation for students. Computer Science Department: The Computer Science department offers courses in programming, database management, data structures, artificial intelligence, machine learning and software engineering. Laboratory: Students must follow laboratory safety rules and complete assigned practical exercises during laboratory sessions. Leave: Students should obtain appropriate permission for planned l

In [55]:
def create_chunks(text, chunk_size=40):
  words= text.split()
  chunks= []
  for i in range(0, len(words), chunk_size):
    chunk= " ".join(words[i:i+chunk_size])
    chunks.append(chunk)
  return(chunks)
chunks= create_chunks(text)
print("number(chunks)", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\nchunk{i+1}:")
    print(chunk[:300])

number(chunks) 4

chunk1:
COLLEGE INFORMATION Attendance Policy: Students must maintain the minimum attendance percentage required by the college to be eligible for semester examinations. Examination: Semester examinations are conducted at the end of each semester. Students must complete the required academic and attendance

chunk2:
requirements before appearing for examinations. Library: The college library provides textbooks, reference books, journals and digital learning resources for students. Placement: The placement cell conducts aptitude training, technical training, communication training and interview preparation for s

chunk3:
Computer Science department offers courses in programming, database management, data structures, artificial intelligence, machine learning and software engineering. Laboratory: Students must follow laboratory safety rules and complete assigned practical exercises during laboratory sessions. Leave: S

chunk4:
planned leave according to college regulation

In [56]:
embedding_model= embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [57]:
chunk_embeddings= embedding_model.encode(chunks, convert_to_numpy=True)
print(chunk_embeddings.shape)

(4, 384)


In [58]:
dimension= chunk_embeddings.shape[1]
index= faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings).astype("float32"))
print(index.ntotal)

4


In [59]:
def retrieve_context(question, top_k=3):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        top_k
    )

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

In [60]:
question= "What are the Library Facilities?"
results= retrieve_context(question)
for i, result in enumerate(results):
  print(f"\nResult {i+1}")
  print(result)


Result 1
requirements before appearing for examinations. Library: The college library provides textbooks, reference books, journals and digital learning resources for students. Placement: The placement cell conducts aptitude training, technical training, communication training and interview preparation for students. Computer Science Department: The

Result 2
Computer Science department offers courses in programming, database management, data structures, artificial intelligence, machine learning and software engineering. Laboratory: Students must follow laboratory safety rules and complete assigned practical exercises during laboratory sessions. Leave: Students should obtain appropriate permission for

Result 3
COLLEGE INFORMATION Attendance Policy: Students must maintain the minimum attendance percentage required by the college to be eligible for semester examinations. Examination: Semester examinations are conducted at the end of each semester. Students must complete the required aca

In [61]:
def build_context(question):
  retrieved_chunks= retrieve_context(question)
  context= "\n\n".join(retrieved_chunks)
  return context

In [62]:
context= build_context("What are the Library facilities")
print(context)

requirements before appearing for examinations. Library: The college library provides textbooks, reference books, journals and digital learning resources for students. Placement: The placement cell conducts aptitude training, technical training, communication training and interview preparation for students. Computer Science Department: The

Computer Science department offers courses in programming, database management, data structures, artificial intelligence, machine learning and software engineering. Laboratory: Students must follow laboratory safety rules and complete assigned practical exercises during laboratory sessions. Leave: Students should obtain appropriate permission for

COLLEGE INFORMATION Attendance Policy: Students must maintain the minimum attendance percentage required by the college to be eligible for semester examinations. Examination: Semester examinations are conducted at the end of each semester. Students must complete the required academic and attendance


In [63]:
def create_prompt(question, context):

    prompt = f"""
You are a College Student Knowledge Assistant.

Answer the student's question using ONLY the
information provided in the context.

If the answer is not available in the context,
say:

"I couldn't find this information in the provided
college documents."

Do not invent information.

Context:
{context}

Student Question:
{question}

Answer clearly and briefly.
"""

    return prompt

In [64]:
def generate_answer(question):

    context = build_context(question)

    prompt = create_prompt(
        question,
        context
    )

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

In [66]:
question= "explain Attendance policy?"
answer= generate_answer(question)
print(question)
print(answer)

explain Attendance policy?
Based on the provided documents, under the Attendance Policy, students must maintain the minimum attendance percentage required by the college to be eligible for semester examinations.


In [67]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello"
)

print(response.text)

Hello! How can I help you today?


In [69]:
questions = [
    "What is the attendance policy?",
    "What does the placement cell provide?",
    "What courses are available in Computer Science?",

]

for question in questions:

    print("\nQuestion:", question)

    answer = generate_answer(question)

    print("Answer:", answer)


Question: What is the attendance policy?
Answer: Students must maintain the minimum attendance percentage required by the college to be eligible for semester examinations.

Question: What does the placement cell provide?
Answer: The placement cell conducts aptitude training, technical training, communication training, and interview preparation for students.

Question: What courses are available in Computer Science?
Answer: The Computer Science department offers courses in:
* Programming
* Database management
* Data structures
* Artificial intelligence
* Machine learning
* Software engineering


In [70]:
def chatbot(question, history):

    if not question.strip():
        return "Please enter a question."

    answer = generate_answer(question)

    return answer

In [71]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="🎓 College Student Knowledge Assistant",
    description="Ask questions based on the college knowledge documents.",
    textbox=gr.Textbox(
        placeholder="Ask your college-related question..."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba42888639d68c6b3e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
